In [7]:
# Data Visualization
#1.prepare data for visualization
#2.anlyiz number of rides by
   # season
   # month
   # compare with weather conditions
# 3.Visualize Top Start Stations
# 4.Visualize Top End Stations
# 5.Homework Part 2

In [1]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [3]:
weather_daily = pd.read_csv('../data/citibike/JC/jersey_weather.csv')
citibike_df = pd.read_csv('../data/citibike/JC/JC2025_Enriched.csv')
citibike_df = pd.read_csv('../data/citibike/JC/JC2026_Enriched.csv')

In [5]:
# Combine CitiBike data
citibike_df = pd.concat(
    [citibike_df, citibike_df],
    ignore_index=True
)

In [6]:
citibike_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 825700 entries, 0 to 825699
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ride_id                825700 non-null  str    
 1   rideable_type          825700 non-null  str    
 2   started_at             825700 non-null  str    
 3   ended_at               825700 non-null  str    
 4   start_station_name     825700 non-null  str    
 5   start_station_id       825700 non-null  str    
 6   end_station_name       825700 non-null  str    
 7   end_station_id         825700 non-null  str    
 8   start_lat              825700 non-null  float64
 9   start_lng              825700 non-null  float64
 10  end_lat                825700 non-null  float64
 11  end_lng                825700 non-null  float64
 12  member_casual          825700 non-null  str    
 13  year                   825700 non-null  int64  
 14  ride_duration_minutes  825700 non-null  float64

In [8]:
citibike_df['date'] = pd.to_datetime(citibike_df['date'], errors='coerce')

In [9]:
weather_daily.info()

<class 'pandas.DataFrame'>
RangeIndex: 559 entries, 0 to 558
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   temperature_2m_max   559 non-null    float64
 1   temperature_2m_min   559 non-null    float64
 2   temperature_2m_mean  559 non-null    float64
 3   precipitation_sum    559 non-null    float64
 4   rain_sum             559 non-null    float64
 5   snowfall_sum         559 non-null    float64
 6   wind_speed_10m_max   559 non-null    float64
 7   date                 559 non-null    str    
dtypes: float64(7), str(1)
memory usage: 35.1 KB


#### Number of rides per monhts

In [10]:
#First, we create a monthly aggregation.
#Each row will represent one month.
monthly_rides = (
    citibike_df
    .groupby("month", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

monthly_rides

,month,number_of_rides
0,2026-01,78406
1,2026-02,50782
2,2026-03,123850
3,2026-04,163690
4,2026-05,189968
5,2026-06,219004


In [11]:
# Visualization the number of rides per month.

fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show()

#### Number of Rides per Season


In [12]:
#Now we aggregate the data by season.
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
2,Winter,129188
0,Spring,477508
1,Summer,219004


In [13]:
#Vizualization by Season
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

#### Top 10 Start Stations


In [14]:
#First, we calculate the top 10 stations by number of departures.
top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(
        number_of_departures=("ride_id", "count")
    )
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations


,start_station_name,number_of_departures
51,Grove St PATH,38162
93,River St & Newark St,29018
58,Hoboken Terminal - Hudson St & Hudson Pl,21826
19,Bergen Ave & Sip Ave,19696
52,Hamilton Park,17882
0,11 St & Washington St,16938
43,Exchange Pl,16250
92,River St & 1 St,15762
95,South Waterfront Walkway - Sinatra Dr & 1 St,15750
33,Columbus Park - Clinton St & 9 St,15654


In [15]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

#### Top 10 End Stations


In [16]:
#Now we calculate the top 10 stations by number of arrivals.
top_end_stations = (
    citibike_df
    .dropna(subset=["end_station_name"])
    .groupby("end_station_name", as_index=False)
    .agg(
        number_of_arrivals=("ride_id", "count")
    )
    .sort_values("number_of_arrivals", ascending=False)
    .head(10)
)

top_end_stations

,end_station_name,number_of_arrivals
162,Grove St PATH,38846
252,River St & Newark St,30436
169,Hoboken Terminal - Hudson St & Hudson Pl,22050
58,Bergen Ave & Sip Ave,19450
163,Hamilton Park,17976
8,11 St & Washington St,17010
140,Exchange Pl,16104
95,Columbus Park - Clinton St & 9 St,15744
264,South Waterfront Walkway - Sinatra Dr & 1 St,15302
11,14 St Ferry - 14 St & Shipyard Ln,15252


In [17]:
fig = px.bar(
    top_end_stations.sort_values("number_of_arrivals"),
    x="number_of_arrivals",
    y="end_station_name",
    orientation="h",
    title="Top 10 End Stations by Number of Arrivals",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Arrivals",
    yaxis_title="End Station"
)

fig.show()

#### Merge Citibike with Daily Rides

In [18]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2026-01-01,1320
1,2026-01-02,1950
2,2026-01-03,2176
3,2026-01-04,2122
4,2026-01-05,3334


In [19]:
weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,11.0,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [20]:
weather_daily["date"] = pd.to_datetime(weather_daily["date"])

weather_daily.dtypes

temperature_2m_max            float64
temperature_2m_min            float64
temperature_2m_mean           float64
precipitation_sum             float64
rain_sum                      float64
snowfall_sum                  float64
wind_speed_10m_max            float64
date                   datetime64[us]
dtype: object

In [21]:
#Now we merge the two datasets.
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2026-01-01,1320,0.7,-7.1,-3.4,1.1,0.0,0.77,25.1
1,2026-01-02,1950,-1.3,-7.3,-4.4,0.3,0.0,0.21,14.4
2,2026-01-03,2176,-0.7,-6.7,-3.5,0.0,0.0,0.00,10.3
3,2026-01-04,2122,1.0,-4.5,-2.2,0.3,0.0,0.21,12.1
4,2026-01-05,3334,0.9,-5.4,-1.8,0.8,0.0,0.56,12.6


In [22]:
#Check missing values after the merge.
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
precipitation_sum      0
rain_sum               0
snowfall_sum           0
wind_speed_10m_max     0
dtype: int64

#### Rides and Average Temperature


In [23]:
import plotly.express as px
import statsmodels.api as sm

In [24]:
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

#### Rides and Wind Speed


In [25]:
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

#### Rides and Precipitation


In [26]:
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

#### Dual y axis | rides vs temperature

In [27]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["number_of_rides"],
        mode="lines",
        name="Daily Rides",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["temperature_2m_mean"],
        mode="lines",
        name="Daily Average Temperature",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Daily Rides and Daily Average Temperature",
    xaxis=dict(
        title="Day"
    ),
    yaxis=dict(
        title="Daily Rides",
        side="left"
    ),
    yaxis2=dict(
        title="Daily Average Temperature",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

In [28]:
# Save combined CitiBike data

citibike_df.to_csv(
    "../data/citibike/JC/JC_Enriched.csv",
    index=False
)